# 01 - Fixed-size chunking

Fixed-size chunking splits text into chunks with the same target number of words. It does not understand sentences, paragraphs, headings, or legal structure. That is exactly why it is useful as a baseline: every later method should prove that it improves something over this simple approach.

In RAG, fixed-size chunking answers this question: _What happens if every chunk has a predictable size?_

## What this notebook teaches

- How to extract text from the shared PDF.
- How to split each page into fixed-size word windows.
- Why fixed-size chunks are easy to compare but can cut evidence in the wrong place.
- What metadata each chunk should carry for RAG citations.

In [ ]:
from pathlib import Path
import re

from pypdf import PdfReader


ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")

PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"


def normalize_text(text):
    """Clean common PDF extraction whitespace without changing the document meaning."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(path):
    """Return one record per PDF page with page number and extracted text."""
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages


def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {
        "chunks": len(chunks),
        "min_words": min(lengths),
        "avg_words": round(sum(lengths) / len(lengths), 1),
        "max_words": max(lengths),
    }


def print_chunks(chunks, limit=5, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']} | words {chunk.get('start_word', '?')}-{chunk.get('end_word', '?')}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")


pages = load_pdf_pages(PDF_PATH)
{
    "pages": len(pages),
    "words": sum(len(page["text"].split()) for page in pages),
    "characters": sum(len(page["text"]) for page in pages),
}

## Fixed-size chunking function

This function works page by page. For each page, it splits the page into words and creates a new chunk every `words_per_chunk` words.

Important behavior:

- `start_word` and `end_word` are word positions, not character positions.
- The last chunk on a page can be smaller than `words_per_chunk`.
- A sentence can be cut in half because the function only counts words.
- Page metadata is preserved so retrieved chunks can be cited.

In [ ]:
def fixed_size_chunks(pages, words_per_chunk=180):
    """Split each PDF page into chunks of at most words_per_chunk words."""
    chunks = []

    for page in pages:
        words = page["text"].split()

        for start in range(0, len(words), words_per_chunk):
            end = min(start + words_per_chunk, len(words))
            chunk_words = words[start:end]

            chunks.append({
                "id": f"fixed-{len(chunks) + 1}",
                "page_start": page["page"],
                "page_end": page["page"],
                "start_word": start,
                "end_word": end,
                "text": " ".join(chunk_words),
            })

    return chunks


chunks = fixed_size_chunks(pages, words_per_chunk=180)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=6)

## When this method is good

Fixed-size chunking is good for a first baseline because it is predictable, fast, and easy to debug.

## When this method is weak

It can split an important legal sentence or paragraph across two chunks. In RAG, that can make the retriever return incomplete evidence. Use the later notebooks to compare whether sentence, paragraph, heading-aware, or semantic chunking produces more useful evidence.